# Chapter 5
Hypothesis 2: 
Back-propagation migration with sign-bit time-reversal is the most 
noise-robust technique for time-lapse phase-plane tracking: it suppresses impulsive Laplace 
noise while Kirchhoff creates false-coherent artefacts and Gazdag adds incoherent speckle

_____
# Chapter 5.0 Setting up the Python Notebook
- load in functions
- load in migration results


In [ ]:
import os, sys, pickle
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # avoid libomp double-init crash (pylops + MKL)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import hilbert
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helper_functions.figures import setup_autosave
from helper_functions.visualisation import (
    plot_bscan_grid, plot_spectrum_grid, plot_method_comparison_grid,
)
from helper_functions.WLS import estimate_shift_2d

# Every plt.show() below is auto-saved to TimeLapse_Figures/Hypothesis_2/<category>/<NNN>_<title>.png
setup_autosave(study="Hypothesis_2", prefix="H2_")

# ── Physical constants (same shared medium as Chapter 4: pure ice) ───────────────────
eps_r   = 3.15
v_ice   = 0.299792458 / np.sqrt(eps_r)   # m/ns
f_c_GHz = 1.5                              # Ricker centre frequency [GHz]
lam     = v_ice / f_c_GHz                  # wavelength [m]
kz_c    = 2 * np.pi / lam                  # central wavenumber for the WLS phase-plane fit
t0_ns   = np.sqrt(2) / f_c_GHz             # Ricker peak delay -- only used for excitation-timing plots below

domain_x, domain_y = 4.0, 1.0
y_surface = 0.9     # air-ice interface [m from domain bottom]
pml_t     = 0.010   # 10 PML cells x 1 mm

# FDTD grid cell size -- "true" displacement gprMax actually realised is the nominal
# fraction-of-lambda shift rounded to the nearest 1 mm cell (see Hypothesis_1.ipynb §4.0
# for the full rationale); applied consistently here too.
FDTD_CELL_M = 0.001

def fdtd_true(nominal_m):
    """Round a nominal (continuous) displacement to the nearest FDTD grid cell (1 mm)."""
    return np.round(np.asarray(nominal_m, dtype=float) / FDTD_CELL_M) * FDTD_CELL_M

METHODS      = ['Kirchhoff', 'Gazdag', 'Back-prop']
_METHOD_KEYS = {'Kirchhoff': 'kirchhoff', 'Gazdag': 'gazdag', 'Back-prop': 'backprop'}

# ── Single source of truth: where every movement type's pre-computed cache lives
# (same registry as Hypothesis_1.ipynb §4.0). Chapter 5 is entirely about the NOISY
# (Laplace-noise-augmented) migrated stacks, so load_study() below only builds the
# noisy dicts -- no clean migrated_diff is needed here (Chapter 4 already covers it),
# but the clean 'migrated'/'diff' archives are still opened where the noisy archive
# itself doesn't carry x_traces/z_img (Diagonal). ─────────────────────────────────────
STUDIES = {
    'Lateral': dict(
        root=ROOT / 'timelapse_study', static='static_results.npz',
        static_noisy='static_results_noisy.npz',
        migrated='migrated_results.npz', diff='difference_migrated_results.npz',
        migrated_noisy='migrated_results_noisy.npz',
        shift_keys=('separation_lambda', None), pos_keys=('x_s1', None),
        axis='x', scale=1.0, is_fluidflow=False,
        target='single cylindrical point scatterer, shifted laterally',
    ),
    'Vertical': dict(
        root=ROOT / 'vertical_timelapse_study', static='vertical_static_results.npz',
        static_noisy='vertical_static_results_noisy.npz',
        migrated='vertical_migrated_results.npz', diff='vertical_difference_migrated_results.npz',
        migrated_noisy='vertical_migrated_results_noisy.npz',
        shift_keys=(None, 'shift_lambda'), pos_keys=(None, 'z_depths'),
        axis='z', scale=1.0, is_fluidflow=False,
        target='single cylindrical point scatterer, shifted vertically (deeper)',
    ),
    'Diagonal': dict(
        root=ROOT / 'diagonal_timelapse_study', static='static_results.npz',
        static_noisy='static_results_noisy.npz',
        migrated='diagonal_migrated_results.npz', diff='diagonal_difference_migrated_results.npz',
        migrated_noisy='migrated_results_noisy.npz',
        shift_keys=('shift_lambda_x', 'shift_lambda_y'), pos_keys=('x_scatterers', 'z_depths'),
        axis='xz', scale=1.0, is_fluidflow=False,
        target='single cylindrical point scatterer, shifted diagonally (2:1 x:z ratio)',
    ),
    'FluidFlow': dict(
        root=ROOT / 'fluidflow_study', static='static_results.npz',
        static_noisy='static_results_noisy.npz',
        migrated='migrated_results.npz', diff='difference_migrated_results.npz',
        migrated_noisy='migrated_results_noisy.npz',
        shift_keys=('separation_lambda', None), pos_keys=('x_scatterer', None),
        axis='x', scale=2.0, is_fluidflow=True,
        target='graded wetting-zone front (9-step permittivity ramp, water -> ice), shifted laterally',
    ),
}


def load_study(name):
    """Load the NOISY migrated stacks (+ their unmigrated noisy B-scans) for one
    movement-type study, normalising the per-study npz schema the same way as
    Hypothesis_1.ipynb's load_study() -- no clean migrated_diff is built here since
    Chapter 5 only ever compares noisy Monitor against noisy Baseline."""
    cfg = STUDIES[name]
    root = cfg['root']
    mig    = np.load(root / cfg['migrated'])
    diff   = np.load(root / cfg['diff'])          # only used as an x_traces/z_img fallback
    mign   = np.load(root / cfg['migrated_noisy'])
    staticn = np.load(root / cfg['static_noisy'])

    labels_all = [str(s) for s in mig['scenarios']]   # incl. 'Baseline'
    labels     = labels_all[1:]                        # shift scenarios only
    n_all      = len(labels_all)

    # diagonal_migrated_results.npz doesn't store the grid axes -- fall back to the diff archive
    x_traces = mig['x_traces'] if 'x_traces' in mig.files else diff['x_traces']
    z_img    = mig['z_img']    if 'z_img'    in mig.files else diff['z_img']

    migrated_noisy = {m: dict(zip(labels_all, mign[_METHOD_KEYS[m]])) for m in METHODS}
    migrated_diff_noisy = {
        m: {lbl: migrated_noisy[m][lbl] - migrated_noisy[m][labels_all[0]] for lbl in labels}
        for m in METHODS
    }

    shift_x_key, shift_z_key = cfg['shift_keys']
    pos_x_key, pos_z_key     = cfg['pos_keys']
    shift_x = mig[shift_x_key] if shift_x_key else np.zeros(n_all)
    shift_z = mig[shift_z_key] if shift_z_key else np.zeros(n_all)

    if pos_x_key:
        raw = mig[pos_x_key]
        marker_x_all = raw if np.ndim(raw) else np.full(n_all, float(raw))
    else:
        marker_x_all = np.full(n_all, float(mig['x_scatterer'] if 'x_scatterer' in mig.files
                                              else mig['x_baseline']))
    marker_z_all = mig[pos_z_key] if pos_z_key else np.full(n_all, float(mig['z_top']))

    study = dict(
        name=name, labels_all=labels_all, labels=labels,
        x_traces=x_traces, z_img=z_img, time_ns=staticn['time_ns'],
        data_static_noisy=list(zip(labels_all, staticn['data_static'])),
        migrated_noisy=migrated_noisy, migrated_diff_noisy=migrated_diff_noisy,
        marker_x_all=marker_x_all, marker_z_all=marker_z_all,
        true_dx=fdtd_true(shift_x[1:] * lam), true_dz=fdtd_true(shift_z[1:] * lam),
        extent_bscan=[x_traces[0], x_traces[-1], staticn['time_ns'][-1], 0],
        extent_mig=[x_traces[0], x_traces[-1], z_img[-1], z_img[0]],
        dz_mig=float(z_img[1] - z_img[0]), dx_mig=float(x_traces[1] - x_traces[0]),
        axis=cfg['axis'], scale=cfg['scale'], is_fluidflow=cfg['is_fluidflow'], target=cfg['target'],
        noise_level=float(mign['noise_level']) if 'noise_level' in mign.files else None,
    )
    return study


DATA = {name: load_study(name) for name in STUDIES}
for _name, _s in DATA.items():
    print(f"{_name:10s}  {len(_s['labels_all'])} scenarios  "
          f"x_traces{_s['x_traces'].shape}  z_img{_s['z_img'].shape}  "
          f"noise_level={_s['noise_level']}")
    print(f"           target: {_s['target']}")

# Accumulators populated by the §5.5-5.8 amplitude/phase cells below, consumed by §5.9
AMPLITUDE_RESULTS_NOISY = {}
PHASE_RESULTS_NOISY = {}


# ══════════════════════════════════════════════════════════════════════════════════
# Shared processing pipeline for §5.5-5.8 -- same cropping / WLS phase-plane fit /
# FWHM/Rayleigh resolution analysis as Hypothesis_1.ipynb §4.1, but every function
# reads from study['migrated_noisy'] / study['migrated_diff_noisy'] instead of the
# clean stacks, since Chapter 5 is entirely about how phase-plane tracking survives
# Laplace noise. Apex-finding, however, deliberately differs from Hypothesis_1: since
# this is synthetic FDTD data, the true scatterer/front position is always known, so
# shared_apex_noisy / fluidflow_base_apex_noisy use that ground truth directly rather
# than hunting for the apex in the noisy envelope, which Gazdag's incoherent noise
# speckle (Sec. 5.2) was regularly spoofing into a wrong, scenario-specific location.
# ══════════════════════════════════════════════════════════════════════════════════

# ── Shared figure-text sizing for the §5.X.2 / §5.X.3 diagnostic figures (mirrors
# Hypothesis_1.ipynb's FS_* constants) ───────────────────────────────────────────────
FS_SUPTITLE = 16
FS_TITLE    = 13
FS_LABEL    = 12
FS_TICK     = 10
FS_LEGEND   = 10
FS_SUMMARY  = 15   # True/Est/Err numeric-summary panel (col 3 of the phase diagnostic)

def shared_apex_noisy(study, method=None, search_lam=None):
    """Ground-truth apex (x, z) for the noisy point-scatterer studies (Lateral/
    Vertical/Diagonal): both the true baseline x position AND the true baseline depth
    are known exactly here (it's FDTD synthetic data, not a field survey), so the crop
    window is centred on (study['marker_x_all'][0], study['marker_z_all'][0]) directly
    rather than hunted for in the noisy Hilbert-envelope difference. That envelope
    search is what previously failed on Gazdag: its phase-shift migration turns noise
    into incoherent speckle (Sec. 5.2), which regularly out-peaks the genuine
    scatterer signal and drags the whole crop window -- and therefore the WLS
    phase-plane fit -- onto the wrong location. Cropping in x alone (keeping the full
    z range) was still letting that speckle elsewhere in depth dominate the fit's
    weighted bins, so the window is now tight in z too -- see crop_bounds_noisy. Using
    the known ground truth sidesteps that failure mode entirely; this shortcut is only
    valid because the true position is available, which it would not be for real
    field data. `method`/`search_lam` are accepted (unused) so every existing call
    site keeps working unchanged."""
    return study['marker_x_all'][0], study['marker_z_all'][0]


def fluidflow_base_apex_noisy(study, method=None):
    """Ground-truth apex (x, z) for the noisy FluidFlow study: the baseline front's
    centre and depth are known exactly (reconstructed from the same true front
    positions used to build the geometry -- see Hypothesis_1.ipynb's load_study()).
    Identical to shared_apex_noisy; kept as a separate name so the is_fluidflow
    branches at each call site read clearly. `method` is accepted (unused) to keep
    this a drop-in replacement for every existing call site."""
    return shared_apex_noisy(study, method)


def crop_bounds_noisy(x_traces, z_img, x_apex, z_apex, crop_hw_lam=2.5):
    """2-D crop index bounds around a known (x, z) apex -- half-width crop_hw_lam*lam
    in BOTH dimensions, not just x. Restricting the crop in z as well as x is what
    keeps Gazdag's noise speckle sitting at other depths (but the same lateral
    position) out of the WLS fit's weighted bins."""
    hw = crop_hw_lam * lam
    ix_lo = np.searchsorted(x_traces, x_apex - hw)
    ix_hi = np.searchsorted(x_traces, x_apex + hw)
    iz_lo = np.searchsorted(z_img, z_apex - hw)
    iz_hi = np.searchsorted(z_img, z_apex + hw)
    return ix_lo, ix_hi, iz_lo, iz_hi


def run_phase_test_noisy(study):
    """WLS phase-plane displacement estimate for every (method, scenario) pair of a
    study, from the NOISY migrated stacks. Every study now uses a single, method-
    level apex/crop shared across all scenarios (shared_apex_noisy for the point-
    scatterer studies, fluidflow_base_apex_noisy for FluidFlow) rather than a
    per-scenario search -- see shared_apex_noisy's docstring for why -- and crops in
    both x and z around that known position (crop_bounds_noisy)."""
    rows = []
    for method in METHODS:
        eff_scale = 1.0 if method == 'Back-prop' else study['scale']

        if study['is_fluidflow']:
            x_apex_base, z_apex_base = fluidflow_base_apex_noisy(study, method)
        else:
            x_apex_base, z_apex_base = shared_apex_noisy(study, method)
        ix_lo, ix_hi, iz_lo, iz_hi = crop_bounds_noisy(study['x_traces'], study['z_img'],
                                                         x_apex_base, z_apex_base)
        base_img = study['migrated_noisy'][method][study['labels_all'][0]]
        base_crop = np.nan_to_num(base_img)[iz_lo:iz_hi, ix_lo:ix_hi]

        for i, lbl in enumerate(study['labels']):
            mon_img = study['migrated_noisy'][method][lbl]
            mon_crop = np.nan_to_num(mon_img)[iz_lo:iz_hi, ix_lo:ix_hi]
            dz_est, dx_est, *_ = estimate_shift_2d(base_crop, mon_crop,
                                                    study['dz_mig'], study['dx_mig'], kz_c)
            true_dz, true_dx = study['true_dz'][i], study['true_dx'][i]
            rows.append(dict(
                movement=study['name'], method=method, scenario=lbl,
                shift_lambda=np.hypot(true_dz, true_dx) / lam,
                true_dz_mm=true_dz * 1e3, true_dx_mm=true_dx * 1e3,
                est_dz_mm=eff_scale * dz_est * 1e3, est_dx_mm=eff_scale * dx_est * 1e3,
            ))
    df = pd.DataFrame(rows)
    df['err_dz_mm'] = df['est_dz_mm'] - df['true_dz_mm']
    df['err_dx_mm'] = df['est_dx_mm'] - df['true_dx_mm']
    return df


def style_error_table(df, value_cols, decimals=2):
    """Diverging red/blue table styling shared by every error table in this notebook
    (white ~ 0 mm error, saturating red/blue at the largest error in the table)."""
    vmax = np.nanmax(np.abs(df[value_cols].to_numpy(dtype=float)))
    vmax = vmax if vmax > 0 else 1.0
    fmt = {c: (lambda v, d=decimals: '-' if pd.isna(v) else f'{v:+.{d}f}') for c in value_cols}
    return df.style.format(fmt).background_gradient(cmap='RdBu_r', subset=value_cols, vmin=-vmax, vmax=vmax)


def style_pct_table(df, value_cols, decimals=1):
    """Same diverging colour scale as style_error_table, but values shown as % of
    true displacement."""
    vmax = np.nanmax(np.abs(df[value_cols].to_numpy(dtype=float)))
    vmax = vmax if vmax > 0 else 1.0
    fmt = {c: (lambda v, d=decimals: '-' if pd.isna(v) else f'{v:+.{d}f} %') for c in value_cols}
    return df.style.format(fmt).background_gradient(cmap='RdBu_r', subset=value_cols, vmin=-vmax, vmax=vmax)


def plot_phase_diagnostic_noisy(study, method, search_lam=3.0, crop_hw_lam=2.5):
    """Noisy-data counterpart of Hypothesis_1.ipynb's plot_phase_diagnostic -- same
    6-column WLS diagnostic (cropped difference image, cross-spectrum phase, cross-
    spectrum energy, numeric summary, plane fit along kx, plane fit along kz), one row
    per scenario. Point-scatterer studies use a single shared apex/crop per method
    (shared_apex_noisy), not a per-scenario search -- see that function's docstring.
    The last two columns detrend the fitted cross-spectrum phase against the *other*
    axis's contribution (phi - kz*dz for the kx panel, phi - kx*dx for the kz panel)
    so the remaining 1-D scatter should collapse onto the fitted line
    phi = k*shift + phi_0 if the plane fit is good; points are coloured by the
    cross-spectrum magnitude |XS|, i.e. the WLS weight each bin actually received."""
    x_traces, z_img = study['x_traces'], study['z_img']
    dz_mig, dx_mig = study['dz_mig'], study['dx_mig']
    base_img = np.nan_to_num(study['migrated_noisy'][method][study['labels_all'][0]])
    eff_scale = 1.0 if method == 'Back-prop' else study['scale']

    if study['is_fluidflow']:
        x_apex_base, z_apex_base = fluidflow_base_apex_noisy(study, method)
    else:
        x_apex_base, z_apex_base = shared_apex_noisy(study, method, search_lam)
    ix_lo0, ix_hi0, iz_lo0, iz_hi0 = crop_bounds_noisy(x_traces, z_img, x_apex_base, z_apex_base, crop_hw_lam)

    n_cases = len(study['labels'])
    fig, axes = plt.subplots(n_cases, 6, figsize=(30, 4.0 * n_cases))
    axes = np.atleast_2d(axes)
    fig.suptitle(f"{study['name']} (Noisy) -- Phase-plane shift estimation -- {method}  |  "
                 f"Baseline vs each scenario", fontsize=FS_SUPTITLE, fontweight='bold', y=1.01)

    for row, lbl in enumerate(study['labels']):
        mon_img = np.nan_to_num(study['migrated_noisy'][method][lbl])
        true_dz, true_dx = study['true_dz'][row], study['true_dx'][row]
        x_apex, z_apex = x_apex_base, z_apex_base
        ix_lo, ix_hi, iz_lo, iz_hi = ix_lo0, ix_hi0, iz_lo0, iz_hi0

        x_crop = x_traces[ix_lo:ix_hi]
        z_crop = z_img[iz_lo:iz_hi]
        base_crop = base_img[iz_lo:iz_hi, ix_lo:ix_hi]
        mon_crop = mon_img[iz_lo:iz_hi, ix_lo:ix_hi]

        dz_est, dx_est, phi_0, XS, kz_ax, kx_ax, fit_points = estimate_shift_2d(
            base_crop, mon_crop, dz_mig, dx_mig, kz_c, return_fit_points=True)
        dz_raw, dx_raw = dz_est, dx_est   # un-scaled fit, matches fit_points/phi_0 -- used for the plane-fit panels
        dz_est *= eff_scale; dx_est *= eff_scale

        XS_s = np.fft.fftshift(XS); kz_s = np.fft.fftshift(kz_ax); kx_s = np.fft.fftshift(kx_ax)
        energy = np.abs(XS_s)
        phi_show = np.where(energy > 0.05 * energy.max(), np.degrees(np.angle(XS_s)), np.nan)
        klim = 1.5 * kz_c
        extent_crop = [x_crop[0], x_crop[-1], z_crop[-1], z_crop[0]]

        ax0 = axes[row, 0]
        diff = mon_crop - base_crop
        vmax = np.nanmax(np.abs(diff)) or 1.0
        ax0.imshow(diff, aspect='auto', extent=extent_crop, cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, origin='upper')
        ax0.axvline(x_apex, color='k', lw=1.2, ls='--', label='shared apex')
        ax0.axhline(z_apex, color='k', lw=1.2, ls='--')
        if study['is_fluidflow']:
            ax0.axvline(x_apex + true_dx, color='r', lw=1.0, ls=':', label='True Front')
            ax0.axvline(x_apex + dx_est, color='g', lw=1.2, ls='--',
                        label=f'Inferred Front ({eff_scale:.0f}x)')
            ax0.legend(fontsize=FS_LEGEND, loc='lower right')
        ax0.set_title(f"{lbl}  (true dz={true_dz*1e3:.1f}, dx={true_dx*1e3:.1f} mm)\n"
                      f"Difference (mon - base)  |  apex @ x={x_apex*100:.1f} cm, "
                      f"z={z_apex*100:.1f} cm", fontsize=FS_TITLE)
        ax0.set_xlabel('x [m]', fontsize=FS_LABEL); ax0.set_ylabel('z [m]', fontsize=FS_LABEL)
        ax0.tick_params(labelsize=FS_TICK)

        ax1 = axes[row, 1]
        im1 = ax1.pcolormesh(kx_s, kz_s, phi_show, cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
        for sgn in (-1, 1):
            ax1.axhline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
            ax1.axvline(sgn * klim, color='k', lw=0.8, ls='--', alpha=0.5)
        ax1.set_title('Cross-spectrum phase [deg]\ndashed = fit band', fontsize=FS_TITLE)
        ax1.set_xlabel('kx [rad/m]', fontsize=FS_LABEL); ax1.set_ylabel('kz [rad/m]', fontsize=FS_LABEL)
        ax1.set_xlim(-klim * 2.5, klim * 2.5); ax1.set_ylim(-klim * 2.5, klim * 2.5)
        ax1.tick_params(labelsize=FS_TICK)
        cb1 = plt.colorbar(im1, ax=ax1, fraction=0.046); cb1.ax.tick_params(labelsize=FS_TICK)

        ax2 = axes[row, 2]
        im2 = ax2.pcolormesh(kx_s, kz_s, energy, cmap='inferno', shading='auto')
        for sgn in (-1, 1):
            ax2.axhline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
            ax2.axvline(sgn * klim, color='w', lw=0.8, ls='--', alpha=0.5)
        ax2.set_title('Cross-spectrum energy |XS|\ndashed = fit band', fontsize=FS_TITLE)
        ax2.set_xlabel('kx [rad/m]', fontsize=FS_LABEL); ax2.set_ylabel('kz [rad/m]', fontsize=FS_LABEL)
        ax2.set_xlim(-klim * 2.5, klim * 2.5); ax2.set_ylim(-klim * 2.5, klim * 2.5)
        ax2.tick_params(labelsize=FS_TICK)
        cb2 = plt.colorbar(im2, ax=ax2, fraction=0.046); cb2.ax.tick_params(labelsize=FS_TICK)

        ax3 = axes[row, 3]; ax3.axis('off')
        txt = (f"True:  dz = {true_dz*1e3:+7.3f} mm\n       dx = {true_dx*1e3:+7.3f} mm\n\n"
               f"Est:   dz = {dz_est*1e3:+7.3f} mm\n       dx = {dx_est*1e3:+7.3f} mm"
               + (f"  ({eff_scale:.0f}x)" if eff_scale != 1.0 else "") + "\n\n"
               f"Err:   dz = {(dz_est-true_dz)*1e3:+.4f} mm\n       dx = {(dx_est-true_dx)*1e3:+.4f} mm\n\n"
               f"Apex @ x = {x_apex*100:.2f} cm\n         z = {z_apex*100:.2f} cm")
        ax3.text(0.05, 0.92, txt, transform=ax3.transAxes, fontsize=FS_SUMMARY, va='top', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

        # Cols 4-5: WLS plane-fit diagnostics -- the fitted plane phi = kz*dz + kx*dx + phi_0
        # collapsed onto each axis separately by subtracting the *other* axis's fitted
        # contribution, so a good fit shows the scattered points hugging the red line.
        # Point colour = |XS|, the actual per-bin weight the WLS fit used.
        kz_pts, kx_pts = fit_points['kz'], fit_points['kx']
        phi_pts, w_pts = fit_points['phi'], fit_points['weight']

        ax4 = axes[row, 4]
        phi_detrend_x = np.degrees(phi_pts - kz_pts * dz_raw)
        sc4 = ax4.scatter(kx_pts, phi_detrend_x, c=w_pts, cmap='viridis', s=10)
        kx_line = np.array([kx_pts.min(), kx_pts.max()])
        ax4.plot(kx_line, np.degrees(dx_raw * kx_line + phi_0), color='red', lw=1.5, label='WLS fit')
        ax4.set_title('Plane fit -- kx direction\n(phase minus kz contribution)', fontsize=FS_TITLE)
        ax4.set_xlabel('kx [rad/m]', fontsize=FS_LABEL)
        ax4.set_ylabel('phase - kz*dz [deg]', fontsize=FS_LABEL)
        ax4.tick_params(labelsize=FS_TICK); ax4.legend(fontsize=FS_LEGEND)
        cb4 = plt.colorbar(sc4, ax=ax4, fraction=0.046)
        cb4.set_label('|XS| weight', fontsize=FS_LABEL); cb4.ax.tick_params(labelsize=FS_TICK)

        ax5 = axes[row, 5]
        phi_detrend_z = np.degrees(phi_pts - kx_pts * dx_raw)
        sc5 = ax5.scatter(kz_pts, phi_detrend_z, c=w_pts, cmap='viridis', s=10)
        kz_line = np.array([kz_pts.min(), kz_pts.max()])
        ax5.plot(kz_line, np.degrees(dz_raw * kz_line + phi_0), color='red', lw=1.5, label='WLS fit')
        ax5.set_title('Plane fit -- kz direction\n(phase minus kx contribution)', fontsize=FS_TITLE)
        ax5.set_xlabel('kz [rad/m]', fontsize=FS_LABEL)
        ax5.set_ylabel('phase - kx*dx [deg]', fontsize=FS_LABEL)
        ax5.tick_params(labelsize=FS_TICK); ax5.legend(fontsize=FS_LEGEND)
        cb5 = plt.colorbar(sc5, ax=ax5, fraction=0.046)
        cb5.set_label('|XS| weight', fontsize=FS_LABEL); cb5.ax.tick_params(labelsize=FS_TICK)

    plt.tight_layout(); plt.show(); plt.close('all')


def plot_phase_diagnostics_noisy(study, methods=METHODS):
    """Run plot_phase_diagnostic_noisy once per migration method."""
    for method in methods:
        plot_phase_diagnostic_noisy(study, method)


def run_phase_test_section_noisy(study):
    """§5.X.3 driver: show the 6-column WLS diagnostic for every method (noisy data),
    then run the WLS phase test, pivot to a scenario x method error table, style it,
    and return (raw results, styled table)."""
    plot_phase_diagnostics_noisy(study)
    df = run_phase_test_noisy(study)
    cols = ['err_dz_mm', 'err_dx_mm'] if study['axis'] == 'xz' else \
           ['err_dz_mm'] if study['axis'] == 'z' else ['err_dx_mm']
    pivot = (df.pivot(index='scenario', columns='method', values=cols) if len(cols) > 1
             else df.pivot(index='scenario', columns='method', values=cols[0]))
    pivot = pivot.loc[study['labels']]
    styled = style_error_table(pivot, pivot.columns.tolist())
    return df, styled


def fwhm_1d(profile, axis_vals):
    """Full Width at Half Maximum of a single-lobed profile around its (possibly
    negative-polarity) peak, linear-interpolated across the half-max crossing on each
    side. Returns (fwhm, x_left, x_right, x_peak)."""
    amp = np.abs(profile)
    peak_idx = int(np.argmax(amp))
    half = amp[peak_idx] / 2.0

    i = peak_idx
    while i > 0 and amp[i] > half:
        i -= 1
    x_left = axis_vals[i] if i == peak_idx else np.interp(
        half, [amp[i], amp[i + 1]], [axis_vals[i], axis_vals[i + 1]])

    i = peak_idx
    while i < len(amp) - 1 and amp[i] > half:
        i += 1
    x_right = axis_vals[i] if i == peak_idx else np.interp(
        half, [amp[i], amp[i - 1]], [axis_vals[i], axis_vals[i - 1]])

    return abs(x_right - x_left), x_left, x_right, axis_vals[peak_idx]


def _extract_profile_window_noisy(study, img, center, half_window):
    """Slice through the RAW noisy migrated image `img` along the study's motion
    axis, cropped to a local window around `center` -- mirrors Hypothesis_1.ipynb's
    _extract_profile_window."""
    if study['axis'] == 'x':
        full_axis = study['x_traces']
        iz = int(np.argmin(np.abs(study['z_img'] - study['marker_z_all'][0])))
        full_profile = img[iz, :]
    elif study['axis'] == 'z':
        full_axis = study['z_img']
        ix = int(np.argmin(np.abs(study['x_traces'] - study['marker_x_all'][0])))
        full_profile = img[:, ix]
    else:  # 'xz'
        cx0, cz0 = study['marker_x_all'][0], study['marker_z_all'][0]
        cx1, cz1 = study['marker_x_all'][-1], study['marker_z_all'][-1]
        dx_dir, dz_dir = cx1 - cx0, cz1 - cz0
        n = np.hypot(dx_dir, dz_dir) or 1.0
        ux, uz = dx_dir / n, dz_dir / n
        full_axis = np.linspace(-3.0 * lam, 3.0 * lam, 200)
        interp = RegularGridInterpolator((study['z_img'], study['x_traces']), img,
                                          bounds_error=False, fill_value=0.0)
        pts = np.stack([cz0 + full_axis * uz, cx0 + full_axis * ux], axis=-1)
        full_profile = interp(pts)

    lo = np.searchsorted(full_axis, center - half_window)
    hi = np.searchsorted(full_axis, center + half_window)
    lo, hi = max(0, lo), min(len(full_axis), hi)
    if hi - lo < 5:
        lo, hi = 0, len(full_axis)
    return full_profile[lo:hi], full_axis[lo:hi]


def amplitude_resolution_table_noisy(study, methods=METHODS, half_window_lam=2.5):
    """Rayleigh-criterion table on the NOISY migrated images -- same algorithm as
    Hypothesis_1.ipynb's amplitude_resolution_table."""
    hw = half_window_lam * lam
    center0 = (0.0 if study['axis'] == 'xz' else
               study['marker_x_all'][0] if study['axis'] == 'x' else study['marker_z_all'][0])
    rows = []
    for method in methods:
        base_img = study['migrated_noisy'][method][study['labels_all'][0]]
        base_profile, axis_vals = _extract_profile_window_noisy(study, base_img, center0, hw)
        peak = np.max(np.abs(base_profile))
        base_n = base_profile / peak if peak > 0 else base_profile
        fwhm, xl, xr, x_peak_base = fwhm_1d(base_n, axis_vals)

        for i, lbl in enumerate(study['labels']):
            mon_img = study['migrated_noisy'][method][lbl]
            mon_profile, _ = _extract_profile_window_noisy(study, mon_img, center0, hw)
            peak_m = np.max(np.abs(mon_profile))
            mon_n = mon_profile / peak_m if peak_m > 0 else mon_profile
            x_peak_mon = axis_vals[int(np.argmax(np.abs(mon_n)))]
            sep = abs(x_peak_mon - x_peak_base)
            rows.append(dict(scenario=lbl, method=method, fwhm_mm=fwhm * 1e3,
                              separation_mm=sep * 1e3, ratio=sep / fwhm if fwhm > 0 else np.nan))
    return pd.DataFrame(rows)


def plot_amplitude_zoom_noisy(study, methods=METHODS, half_window_lam=2.5):
    """Zoomed Baseline-vs-Monitor PSF grid on the NOISY migrated images, with FWHM
    shading -- same layout as Hypothesis_1.ipynb's plot_amplitude_zoom."""
    labels, hw = study['labels'], half_window_lam * lam
    n_s, n_m = len(labels), len(methods)
    center0 = (0.0 if study['axis'] == 'xz' else
               study['marker_x_all'][0] if study['axis'] == 'x' else study['marker_z_all'][0])

    fig, axes = plt.subplots(n_s, n_m, figsize=(5.6 * n_m, 3.8 * n_s), squeeze=False)
    for j, method in enumerate(methods):
        base_img = study['migrated_noisy'][method][study['labels_all'][0]]
        base_profile, axis_vals = _extract_profile_window_noisy(study, base_img, center0, hw)
        peak = np.max(np.abs(base_profile))
        base_n = base_profile / peak if peak > 0 else base_profile
        fwhm, xl, xr, x_peak_base = fwhm_1d(base_n, axis_vals)

        for i, lbl in enumerate(labels):
            ax = axes[i, j]
            mon_img = study['migrated_noisy'][method][lbl]
            mon_profile, _ = _extract_profile_window_noisy(study, mon_img, center0, hw)
            peak_m = np.max(np.abs(mon_profile))
            mon_n = mon_profile / peak_m if peak_m > 0 else mon_profile
            x_peak_mon = axis_vals[int(np.argmax(np.abs(mon_n)))]
            sep_mm = abs(x_peak_mon - x_peak_base) * 1e3
            ratio = sep_mm / (fwhm * 1e3) if fwhm > 0 else np.nan

            ax.plot(axis_vals, base_n, color='steelblue', lw=1.2, label='Baseline')
            ax.plot(axis_vals, mon_n, color='tomato', lw=1.1, label=lbl)
            ax.axhline(0.5, color='grey', lw=0.6, ls=':'); ax.axhline(-0.5, color='grey', lw=0.6, ls=':')
            ax.axvspan(xl, xr, color='steelblue', alpha=0.12)
            ax.axvline(x_peak_base, color='steelblue', lw=0.8, ls='--')
            ax.axvline(x_peak_mon, color='tomato', lw=0.8, ls='--')
            ax.set_ylim(-1.3, 1.3)
            ax.set_title(f'sep={sep_mm:.2f} mm  FWHM={fwhm*1e3:.1f} mm  ratio={ratio:.2f}', fontsize=FS_TITLE)
            ax.tick_params(labelsize=FS_TICK)
            if i == 0:
                ax.text(0.5, 1.32, method, transform=ax.transAxes, ha='center',
                        fontsize=FS_SUPTITLE - 2, fontweight='bold')
            if j == 0:
                ax.set_ylabel(lbl, fontsize=FS_LABEL, rotation=0, ha='right', va='center')

    fig.suptitle(f"{study['name']} (Noisy) -- Amplitude PSF zoom: Baseline vs Monitor (RAW migrated "
                 f"images)\nshaded = Baseline FWHM  |  dashed = peak positions", fontsize=FS_SUPTITLE, y=1.02)
    plt.tight_layout(h_pad=3.0, w_pad=1.5); plt.show(); plt.close('all')


def run_amplitude_test_noisy(study, methods=METHODS):
    """§5.X.2 driver: zoomed Baseline-vs-Monitor PSF grid from the NOISY migrated
    images with FWHM shading, and the resulting separation/FWHM resolution table."""
    plot_amplitude_zoom_noisy(study, methods)

    table = amplitude_resolution_table_noisy(study, methods)
    pivot = table.pivot(index='scenario', columns='method', values='ratio').loc[study['labels']]
    styled = pivot.style.format('{:.3f}').background_gradient(cmap='Blues_r', axis=None)
    return table, styled


def plot_migration_results_noisy(study, methods=METHODS, gazdag_vmax_percentile=90):
    """§5.X.1 driver: ONE compiled, zoomed grid (rows=scenarios, cols=methods) of the
    NOISY TimeLapse-difference images -- the clean-data comparison already lives in
    Hypothesis_1.ipynb §4.X.3, so only the noisy grid is shown here.

    Each column already gets its own colour scale (plot_method_comparison_grid),
    but Gazdag's own noisy column still washed out to flat white at the default
    100th-percentile scale: its handful of extreme-amplitude speckle pixels
    (max/median ratio ~580 in a representative check, vs. ~270 for Kirchhoff and
    ~63 for Back-prop) stretch the column's scale so far that the genuine,
    ~20x-weaker coherent difference signal renders as indistinguishable from zero.
    gazdag_vmax_percentile=90 clips Gazdag's own scale to a far more robust
    percentile (sacrificing the outlier speckle pixels, which saturate, in
    exchange for making the real signal visible); Kirchhoff/Back-prop keep the
    default 100 since they don't have this outlier problem and a shared low
    percentile over-saturates their already-good-SNR images instead. The
    clean-data comparison in Hypothesis_1.ipynb has no such outliers either and
    keeps the default 100 throughout."""
    vmax_percentile = [gazdag_vmax_percentile if m == 'Gazdag' else 100 for m in methods]
    xs, zs = study['marker_x_all'], study['marker_z_all']
    x_lo_data, x_hi_data = study['x_traces'][0], study['x_traces'][-1]
    z_lo_data, z_hi_data = study['z_img'][0], study['z_img'][-1]
    pad_x = max(3 * lam, 0.5 * (xs.max() - xs.min()))
    pad_z = max(3 * lam, 0.5 * (zs.max() - zs.min()))
    xlim = (max(xs.min() - pad_x, x_lo_data), min(xs.max() + pad_x, x_hi_data))
    ylim = (min(zs.max() + pad_z, z_hi_data), max(zs.min() - pad_z, z_lo_data))

    imgs = [[study['migrated_diff_noisy'][m].get(lbl) for m in methods] for lbl in study['labels']]
    plot_method_comparison_grid(
        imgs, study['extent_mig'], methods, study['labels'], envelope=False,
        marker_x=lambda i, j: study['marker_x_all'][i + 1],
        marker_z=lambda i, j: study['marker_z_all'][i + 1],
        marker_x_baseline=lambda i, j: study['marker_x_all'][0],
        marker_z_baseline=lambda i, j: study['marker_z_all'][0],
        xlim=xlim, ylim=ylim, vmax_percentile=vmax_percentile,
        title=f"{study['name']} -- TimeLapse Migration Comparison (Noisy) -- Signed Amplitude",
    )
    plt.show()
    plt.close('all')

print('Noisy-data workflow functions ready:', ', '.join([
    'plot_migration_results_noisy', 'run_amplitude_test_noisy', 'run_phase_test_section_noisy',
]))

_____
# Chapter 5.1: Noise Creation

In [ ]:
# ── §5.1 Noise Creation: how the Laplace-noise model was derived from gprMax's own
# per-processing-stage noise floor (mirrors Noise_Playground.ipynb's noise-fitting
# cells) -- this is the model actually injected into every '*_noisy.npz' dataset
# used from §5.2 onward. ──────────────────────────────────────────────────────────
with open(ROOT / 'noise_samples.pkl', 'rb') as f:
    noise_samples = pickle.load(f)
with open(ROOT / 'laplace_noise_model.pkl', 'rb') as f:
    laplace_noise_model = pickle.load(f)
with open(ROOT / 'laplace_noise_model_pre_gain.pkl', 'rb') as f:
    laplace_noise_model_pre_gain = pickle.load(f)

stage_names = list(noise_samples.keys())
print(f"{len(stage_names)} processing stages sampled for their intrinsic noise floor:")
for name in stage_names:
    arr = np.asarray(noise_samples[name])
    print(f"  {name:25s} n={arr.size:>8d}  mean={arr.mean():>10.4f}  std={arr.std():>10.4f}")

# ── Distribution per stage, with a Gaussian overlay for reference ────────────────
ncols = 3
nrows = int(np.ceil(len(stage_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, name in zip(axes, stage_names):
    arr = np.asarray(noise_samples[name]).ravel()
    mu, sigma = arr.mean(), arr.std()
    ax.hist(arr, bins=200, density=True, color='steelblue', alpha=0.7, label='samples')
    x = np.linspace(arr.min(), arr.max(), 400)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=1.5, label='Gaussian fit')
    ax.set_title(name, fontsize=10); ax.set_xlabel('amplitude'); ax.set_ylabel('density')
for ax in axes[len(stage_names):]:
    ax.axis('off')
axes[0].legend(fontsize=8)
fig.suptitle('Noise amplitude distribution by processing stage', y=1.02, fontsize=13)
fig.tight_layout(); plt.show(); plt.close('all')

# ── Laplace vs Gaussian fit at the pre-gain stage -- this (not the post-gain
# laplace_noise_model above) is the model actually sampled to build every
# '*_noisy.npz' dataset used from §5.2 onward (see the noise-injection cell below,
# which reads loc/scale from laplace_noise_model_pre_gain) ───────────────────────
final_stage = laplace_noise_model_pre_gain['stage']
arr_final = np.asarray(noise_samples[final_stage]).ravel()
mu_g, sigma_g = arr_final.mean(), arr_final.std()

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(arr_final, bins=200, density=True, color='steelblue', alpha=0.7, label='samples')
x = np.linspace(arr_final.min(), arr_final.max(), 400)
ax.plot(x, stats.laplace.pdf(x, laplace_noise_model_pre_gain['loc'], laplace_noise_model_pre_gain['scale']),
        'g-', lw=1.5, label='Laplace fit')
ax.plot(x, stats.norm.pdf(x, mu_g, sigma_g), 'r--', lw=1.5, label='Gaussian fit')
ax.set_title(f'Noise distribution at "{final_stage}" (pre-gain) with Laplace fit')
ax.set_xlabel('amplitude'); ax.set_ylabel('density'); ax.legend()
fig.tight_layout(); plt.show(); plt.close('all')

print(f"\nAdopted noise model: {laplace_noise_model_pre_gain['distribution']} "
      f"(loc={laplace_noise_model_pre_gain['loc']:.4f}, scale={laplace_noise_model_pre_gain['scale']:.4f}), "
      f"fit from stage '{final_stage}' (n={laplace_noise_model_pre_gain['n_samples_fit']})")
print("The Laplace fit is visibly heavier-tailed than the Gaussian -- consistent with impulsive, "
      "spike-like noise rather than diffuse thermal noise. This is the distribution sampled to "
      "build every '*_noisy.npz' dataset used from §5.2 onward (NOISE_LEVEL scales its 'scale' "
      "parameter to a fixed fraction of each study's Baseline signal std).")

_____
# Chapter 5.2 Migrating noise 

In [ ]:
# ── §5.2 Migrating pure noise: sanity-check the "noise floor" of each migration
# method on an empty, scatterer-free B-scan + Laplace noise only (mirrors
# Noise_Playground.ipynb's "Migrating Pure Noise" section). If a method turns
# incoherent noise into something that looks like a real reflector, that's a
# false-positive risk for every '*_noisy' result used from §5.5 onward. No gprMax
# rerun: Kirchhoff/Gazdag are pure post-processing, and the back-propagation
# snapshots below were already generated once (noise_study/backprop/) and are only
# loaded here. ─────────────────────────────────────────────────────────────────────
from helper_functions.migration import PylopsKirchoffMigration, gazdag_migration

_static_clean = np.load(ROOT / 'timelapse_study' / 'static_results.npz', allow_pickle=False)
ref_signal_std = float(_static_clean['data_static'][0].std())   # clean Baseline, for noise scaling only

x_traces_ref = DATA['Lateral']['x_traces']
z_img_ref    = DATA['Lateral']['z_img']
time_ns_ref  = DATA['Lateral']['time_ns']
dt_ns_ref    = float(time_ns_ref[1] - time_ns_ref[0])
n_t_ref, n_traces_ref = len(time_ns_ref), len(x_traces_ref)

NOISE_LEVEL = 0.1   # same convention as every '*_noisy.npz' dataset used elsewhere in this project
rng = np.random.default_rng(0)
target_std   = NOISE_LEVEL * ref_signal_std
scaled_scale = target_std / np.sqrt(2)   # Var(Laplace) = 2 * scale**2
empty_bscan  = np.zeros((n_t_ref, n_traces_ref))
pure_noise_bscan = empty_bscan + stats.laplace.rvs(
    loc=laplace_noise_model_pre_gain['loc'], scale=scaled_scale,
    size=empty_bscan.shape, random_state=rng,
)

ANGLE_AP = 40   # same Kirchhoff aperture used throughout this project
bscan_tr_first = pure_noise_bscan.T   # Kirchhoff wants (n_tr, n_t); Gazdag wants (n_t, n_x)
mig_kirchhoff_noise = PylopsKirchoffMigration(
    bscan_tr_first, time_ns_ref, x_traces_ref, v_ice, z_img_ref, f0=f_c_GHz, angleaperture=ANGLE_AP)
mig_gazdag_noise = gazdag_migration(pure_noise_bscan, x_traces_ref, time_ns_ref, z_img_ref, v_ice)

extent_bscan_ref = [x_traces_ref[0], x_traces_ref[-1], time_ns_ref[-1], 0]
extent_mig_ref   = [x_traces_ref[0], x_traces_ref[-1], z_img_ref[-1], z_img_ref[0]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
vmax_b = np.abs(pure_noise_bscan).max()
im0 = axes[0].imshow(pure_noise_bscan, aspect='auto', extent=extent_bscan_ref, cmap='seismic',
                      vmin=-vmax_b, vmax=vmax_b, interpolation='nearest')
axes[0].set_title('Pure-noise B-scan (input)'); axes[0].set_xlabel('x [m]'); axes[0].set_ylabel('Time [ns]')
fig.colorbar(im0, ax=axes[0], label='Ez [a.u.]')

vmax_k = np.abs(mig_kirchhoff_noise).max()
im1 = axes[1].imshow(mig_kirchhoff_noise, aspect='auto', extent=extent_mig_ref, cmap='seismic',
                      vmin=-vmax_k, vmax=vmax_k, origin='upper')
axes[1].set_title('Kirchhoff migration of pure noise'); axes[1].set_xlabel('x [m]'); axes[1].set_ylabel('Depth z [m]')
fig.colorbar(im1, ax=axes[1], label='Ez [a.u.]')

vmax_g = np.abs(mig_gazdag_noise).max()
im2 = axes[2].imshow(mig_gazdag_noise, aspect='auto', extent=extent_mig_ref, cmap='seismic',
                      vmin=-vmax_g, vmax=vmax_g, origin='upper')
axes[2].set_title('Gazdag migration of pure noise'); axes[2].set_xlabel('x [m]'); axes[2].set_ylabel('Depth z [m]')
fig.colorbar(im2, ax=axes[2], label='Ez [a.u.]')

fig.suptitle('Migrating Pure Noise (no scatterers, no signal) -- Kirchhoff vs Gazdag',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close('all')

print("Kirchhoff's delay-and-sum aperture stacking imposes coherence on incoherent input by "
      "construction, smoothing pure noise into wave-like bands that could be misread as a real "
      "reflector. Gazdag's phase-shift migration leaves pure noise looking like speckle -- no "
      "wave-like artefacts. Same input noise, different false-positive risk.")

# ── Back-propagation of pure noise, default (peak-normalised) excitation -- the .in
# file was generated and run through gprMax once already
# (noise_study/backprop/purenoise_peaknorm); loaded here from its cached .vti
# snapshots, not regenerated. ─────────────────────────────────────────────────────
import pyvista as pv

def load_pure_noise_focus_frame(slug, n_t, dt_ns, t0_ns, n_snap=30, snap_win=1.0):
    T_ns = n_t * dt_ns
    t_focus_ns = T_ns - t0_ns
    t_start_ns = max(0.0, t_focus_ns - snap_win)
    dt_s = dt_ns * 1e-9
    snap_step = max(1, int((T_ns * 1e-9 - t_start_ns * 1e-9) / (max(1, n_snap - 1) * dt_s)))
    snap_dir = ROOT / 'noise_study' / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns
    idx_focus = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4, len(snap_files) - 1)
    mesh = pv.read(str(snap_files[idx_focus]))
    e_data = np.array(mesh['E-field'])
    return {'ez': e_data[:, 2].reshape(1000, 4000), 't_actual': snap_times_ns[idx_focus]}

focus_peaknorm = load_pure_noise_focus_frame('purenoise_peaknorm', n_t_ref, dt_ns_ref, t0_ns)

fig, ax = plt.subplots(figsize=(7, 6))
vmax = np.percentile(np.abs(focus_peaknorm['ez']), 99.5)
im = ax.imshow(focus_peaknorm['ez'], aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
               extent=[0, 4.0, 0, 1], origin='lower')
ax.set_title(f"Back-propagation of pure noise -- peak-normalised excitation\nt={focus_peaknorm['t_actual']:.2f} ns")
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
fig.colorbar(im, ax=ax, label='Ez [V/m]')
plt.tight_layout(); plt.show(); plt.close('all')

print("With no scatterer present, there is no true location for either excitation scheme to focus "
      "on, so the back-propagated wavefield stays diffuse speckle throughout the domain rather than "
      "collapsing into an obvious spurious bright spot -- unlike Kirchhoff's coherent bands above, "
      "there is no clearly visible artefact here to point at. What does stand out is the amplitude "
      f"scale: peak-normalised excitation reaches only {np.abs(focus_peaknorm['ez']).max():.0f} V/m, "
      "since only each trace's single largest sample is normalised to +-1 and every other noise "
      "sample stays small. §5.3 revisits this with sign-bit excitation, where every sample -- not just "
      "the peak -- is forced to +-1.")


In [ ]:
# ── §5.2 (cont.): spectral content by method -- does migration/back-propagation
# reshape pure noise's frequency content towards the Ricker centre frequency
# (a false-positive risk), or does it stay a flat noise floor like the input?
# Each domain uses its own natural sample axis (time for the raw B-scan, depth
# for the two migrated images, y for the back-propagated snapshot) and is
# converted to an equivalent frequency via f = v_ice * k, the same one-way
# time<->depth mapping already used throughout this notebook (e.g. kz_c above)
# -- this puts all four spectra on one shared, physically comparable axis.
dz_noise = float(z_img_ref[1] - z_img_ref[0])

freqs_noise_ghz = np.fft.rfftfreq(n_t_ref, d=dt_ns_ref)
spec_noise = np.abs(np.fft.rfft(pure_noise_bscan, axis=0, norm='forward')).mean(axis=1)

freqs_kirchhoff_ghz = np.fft.rfftfreq(mig_kirchhoff_noise.shape[0], d=dz_noise) * v_ice
spec_kirchhoff = np.abs(np.fft.rfft(mig_kirchhoff_noise, axis=0, norm='forward')).mean(axis=1)

freqs_gazdag_ghz = np.fft.rfftfreq(mig_gazdag_noise.shape[0], d=dz_noise) * v_ice
spec_gazdag = np.abs(np.fft.rfft(mig_gazdag_noise, axis=0, norm='forward')).mean(axis=1)

freqs_backprop_ghz = np.fft.rfftfreq(focus_peaknorm['ez'].shape[0], d=FDTD_CELL_M) * v_ice
spec_backprop = np.abs(np.fft.rfft(focus_peaknorm['ez'], axis=0, norm='forward')).mean(axis=1)

fig, ax = plt.subplots(figsize=(9, 5.5))
for freqs, spec, label, color in [
    (freqs_noise_ghz, spec_noise, 'Pure noise (input B-scan)', 'grey'),
    (freqs_kirchhoff_ghz, spec_kirchhoff, 'Kirchhoff migration', 'C0'),
    (freqs_gazdag_ghz, spec_gazdag, 'Gazdag migration', 'C1'),
    (freqs_backprop_ghz, spec_backprop, 'Back-propagation (peak-norm)', 'C2'),
]:
    ax.plot(freqs, spec / spec.max(), lw=1.3, color=color, label=label)
ax.axvline(f_c_GHz, color='k', lw=0.8, ls='--', alpha=0.6, label=f'Ricker fc = {f_c_GHz:.1f} GHz')
ax.set_xlim(0, 5); ax.set_xlabel('Frequency [GHz]'); ax.set_ylabel('Normalized |FFT| [-]')
ax.set_title('Migrating Pure Noise -- Spectral Content by Method  (axes rescaled to an equivalent frequency via f = v_ice * k)', fontsize=11)
ax.legend(fontsize=9)
fig.tight_layout(); plt.show(); plt.close('all')

print("All four spectra stay broadly flat noise floors, with no strong energy build-up at f_c -- "
      "none of the three methods manufactures a coherent, wavelet-like frequency peak out of pure "
      "noise on its own. This is a purely spectral view though: it discards phase/spatial-alignment "
      "information, so it does not capture the spatial coherence Kirchhoff's aperture stacking "
      "imposes (visible as wave-like bands in the image above) -- Kirchhoff's false-positive risk is "
      "a spatial-coherence effect, not a frequency-content one.")

_____
# Chapter 5.3 Making back-propagation noise robust

In [ ]:
# ── §5.3 Making back-propagation noise-robust: sign-bit time reversal ───────────────
# Spatial focusing during back-propagation is governed almost entirely by *phase*
# (zero-crossings), not amplitude -- so instead of injecting the peak-normalised
# time-reversed wavefield u(x,tau), only its sign is injected:
#     u_sign(x, tau) = sign(u(x, tau))
# This keeps every zero-crossing/phase trend of the GPR wavelet intact while
# squashing the Laplace noise spikes down to the same +-1 as the coherent signal,
# stripping them of the outsized amplitude that let them dominate the back-
# propagated wavefield (mirrors TimeLapse_Playground.ipynb's noisy back-propagation
# section, cells 86-87; helper_functions.migration.write_backprop_files(...,
# sign_bit=True) implements it). Every Back-prop result used from §5.5 onward
# (DATA[study]['migrated_noisy']['Back-prop']) was already generated with sign-bit
# excitation -- this section only re-derives the excitation-signal transform for
# visualisation, no gprMax rerun. ────────────────────────────────────────────────────

def make_end_taper(n_samples, taper_samples):
    """Unity everywhere, then half-cosine 1->0 over the last `taper_samples`."""
    win = np.ones(n_samples)
    if taper_samples > 0:
        ramp = 0.5 * (1 + np.cos(np.pi * np.arange(taper_samples) / taper_samples))
        win[-taper_samples:] = ramp
    return win

def make_decay_taper(time_ns, tau_ns):
    """Exponential decay starting at 1.0 (t=0) with time constant tau_ns."""
    return np.exp(-time_ns / tau_ns)

def preprocess(data_nt_ntr, dt_ns, time_ns, taper_end_ns=14.0, taper_decay_ns=1.0):
    """Taper a background-subtracted B-scan the same way as every back-propagation
    excitation file in this project (mirrors TimeLapse_Playground.ipynb's preprocess,
    tapering step only -- the t0-shift is not needed for this visualisation)."""
    bscan = data_nt_ntr.T.copy()   # -> (n_tr, n_t)
    n_t = bscan.shape[1]
    w_end = make_end_taper(n_t, int(round(taper_end_ns / dt_ns)))
    w_decay = make_decay_taper(time_ns, taper_decay_ns)
    return bscan * (w_end * w_decay)[np.newaxis, :]

lateral = DATA['Lateral']
sign_bit_bscans = {}
for lbl, raw in lateral['data_static_noisy']:
    tapered = preprocess(raw, dt_ns_ref, lateral['time_ns'])
    sign_bit_bscans[lbl] = np.sign(tapered)   # (n_src, n_t)

freqs_ghz = np.fft.rfftfreq(len(lateral['time_ns']), d=dt_ns_ref)
fig, axes = plt.subplots(2, len(sign_bit_bscans), figsize=(25, 8))
plot_bscan_grid(
    [(f'{lbl} (sign-bit)', sb.T) for lbl, sb in sign_bit_bscans.items()],
    lateral['x_traces'], lateral['time_ns'], axes=axes[0], vmax=1.0, shared_colorbar=False,
)
spectra_sign_bit = [[
    (f'{lbl} (sign-bit)', np.abs(np.fft.rfft(sb.T, axis=0, norm='forward')).mean(axis=1))
    for lbl, sb in sign_bit_bscans.items()
]]
plot_spectrum_grid(spectra_sign_bit, freqs_ghz, title=None, xlim=(0, 5), axes=axes[1])
fig.suptitle('Sign-Bit Time-Reversed Excitation -- B-scans and Spectra (Lateral, Noisy)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout(); plt.show(); plt.close('all')

# ── Direct evidence the fix works: back-propagation of PURE noise (the hardest
# possible case -- zero signal), peak-normalised vs sign-bit excitation, both
# already run through gprMax once in Noise_Playground.ipynb (noise_study/backprop/
# purenoise_signbit) and loaded here from their cached snapshots. ────────────────
focus_signbit = load_pure_noise_focus_frame('purenoise_signbit', n_t_ref, dt_ns_ref, t0_ns)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, frame, title in zip(axes, [focus_peaknorm, focus_signbit],
                             ['Peak-normalised excitation', 'Sign-bit excitation']):
    vmax = np.percentile(np.abs(frame['ez']), 99.5)
    im = ax.imshow(frame['ez'], aspect='auto', cmap='seismic', vmin=-vmax, vmax=vmax,
                   extent=[0, 4.0, 0, 1], origin='lower')
    ax.set_title(f"{title}\nt={frame['t_actual']:.2f} ns")
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    fig.colorbar(im, ax=ax, label='Ez [V/m]')
fig.suptitle('Back-Propagation of Pure Noise -- Peak-Normalised vs Sign-Bit Excitation',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show(); plt.close('all')

print(f"Sign-bit excitation reaches {np.abs(focus_signbit['ez']).max():.0f} V/m, "
      f"~{np.abs(focus_signbit['ez']).max() / np.abs(focus_peaknorm['ez']).max():.0f}x larger than "
      "peak-normalised's, since forcing every sample (not just each trace's single peak) to +-1 "
      "injects far more total energy into the medium -- but the wavefield itself still stays diffuse "
      "speckle in both panels above, exactly as in §5.2: pure noise has no true target to focus on, "
      "so this idealised zero-signal test cannot visually demonstrate whether sign-bit suppresses "
      "spurious focusing the way it can on real data (which does have a true target -- see "
      "TimeLapse_Playground.ipynb cells 86-87). The actual, quantitative evidence that sign-bit "
      "back-propagation is noise-robust comes from §5.9's master MAE table on the real noisy "
      "studies below, where Back-prop achieves the lowest mean-absolute displacement error of the "
      "three methods -- not from this qualitative pure-noise snapshot.")


_____
# Chapter 5.4 Extra processing steps in phase domain to remove noise

In [ ]:
# -- Chapter 5.4: OLS vs. WLS phase-plane fitting on noisy Lateral data --
# estimate_shift_2d fits phi(kz,kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum
# of Baseline vs. Monitor, weighting each masked bin by its cross-spectrum
# magnitude |XS| (WLS) so noisy/incoherent bins contribute less. This section
# demonstrates why that weighting matters by re-running the exact same fit with
# every masked bin given equal weight instead (OLS) -- on Gazdag, whose
# phase-shift migration turns noise into incoherent speckle (Sec. 5.2), the
# smallest sub-wavelength Lateral shift scenario is the hardest realistic case:
# real signal is weakest relative to the noise floor here, so a biased OLS fit
# should show up clearly (larger, multi-wavelength shifts alias the phase-plane
# fit regardless of weighting, and would confound the comparison).
study_54  = DATA['Lateral']
method_54 = 'Kirchhoff'
lbl_54    = study_54['labels'][4]   # '1/32 lambda' -- ~4 mm, the smallest (hardest) real shift

x_apex_54, z_apex_54 = shared_apex_noisy(study_54, method_54)
ix_lo_54, ix_hi_54, iz_lo_54, iz_hi_54 = crop_bounds_noisy(
    study_54['x_traces'], study_54['z_img'], x_apex_54, z_apex_54)

base_crop_54 = np.nan_to_num(study_54['migrated_noisy'][method_54][study_54['labels_all'][0]])[iz_lo_54:iz_hi_54, ix_lo_54:ix_hi_54]
mon_crop_54  = np.nan_to_num(study_54['migrated_noisy'][method_54][lbl_54])[iz_lo_54:iz_hi_54, ix_lo_54:ix_hi_54]

true_dz_54 = study_54['true_dz'][6]
true_dx_54 = study_54['true_dx'][6]

fits_54 = {}
for fit_name, use_weights in [('OLS (unweighted)', False), ('WLS (weighted)', True)]:
    dz_e, dx_e, phi0_e, XS_e, kz_ax_e, kx_ax_e, pts_e = estimate_shift_2d(
        base_crop_54, mon_crop_54, study_54['dz_mig'], study_54['dx_mig'], kz_c,
        weighted=use_weights, return_fit_points=True)
    fits_54[fit_name] = dict(dz_est=dz_e, dx_est=dx_e, phi_0=phi0_e, XS=XS_e,
                              kz_ax=kz_ax_e, kx_ax=kx_ax_e, pts=pts_e)

klim_54 = 1.5 * kz_c
kz_s_54 = np.fft.fftshift(fits_54['WLS (weighted)']['kz_ax'])
kx_s_54 = np.fft.fftshift(fits_54['WLS (weighted)']['kx_ax'])
KZ_s_54, KX_s_54 = np.meshgrid(kz_s_54, kx_s_54, indexing='ij')

fig, axes = plt.subplots(2, 5, figsize=(28, 9.5))

for row, fit_name in enumerate(['OLS (unweighted)', 'WLS (weighted)']):
    f = fits_54[fit_name]
    XS_s = np.fft.fftshift(f['XS'])
    energy = np.abs(XS_s)
    phi_show = np.where(energy > 0.05 * energy.max(), np.degrees(np.angle(XS_s)), np.nan)

    # -- Panel 0: cross-spectrum phase --
    ax0 = axes[row, 0]
    im0 = ax0.pcolormesh(kx_s_54, kz_s_54, phi_show, cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
    for sgn in (-1, 1):
        ax0.axhline(sgn * klim_54, color='k', lw=0.8, ls='--', alpha=0.5)
        ax0.axvline(sgn * klim_54, color='k', lw=0.8, ls='--', alpha=0.5)
    ax0.set_title('Cross-spectrum phase [deg]', fontsize=9)
    ax0.set_xlabel('kx [rad/m]'); ax0.set_ylabel('kz [rad/m]')
    ax0.set_xlim(-klim_54 * 2.5, klim_54 * 2.5); ax0.set_ylim(-klim_54 * 2.5, klim_54 * 2.5)
    plt.colorbar(im0, ax=ax0, fraction=0.046)

    # -- Panel 1: cross-spectrum power --
    ax1 = axes[row, 1]
    im1 = ax1.pcolormesh(kx_s_54, kz_s_54, energy, cmap='inferno', shading='auto')
    for sgn in (-1, 1):
        ax1.axhline(sgn * klim_54, color='w', lw=0.8, ls='--', alpha=0.5)
        ax1.axvline(sgn * klim_54, color='w', lw=0.8, ls='--', alpha=0.5)
    ax1.set_title('Cross-spectrum power |XS|', fontsize=9)
    ax1.set_xlabel('kx [rad/m]'); ax1.set_ylabel('kz [rad/m]')
    ax1.set_xlim(-klim_54 * 2.5, klim_54 * 2.5); ax1.set_ylim(-klim_54 * 2.5, klim_54 * 2.5)
    plt.colorbar(im1, ax=ax1, fraction=0.046)

    # -- Panel 2: fitted plane, same window/colour-scale as panel 0 --
    ax2 = axes[row, 2]
    plane_deg = np.degrees((KZ_s_54 * f['dz_est'] + KX_s_54 * f['dx_est'] + f['phi_0'] + np.pi) % (2 * np.pi) - np.pi)
    plane_show = np.where(energy > 0.05 * energy.max(), plane_deg, np.nan)
    im2 = ax2.pcolormesh(kx_s_54, kz_s_54, plane_show, cmap='RdBu_r', vmin=-180, vmax=180, shading='auto')
    for sgn in (-1, 1):
        ax2.axhline(sgn * klim_54, color='k', lw=0.8, ls='--', alpha=0.5)
        ax2.axvline(sgn * klim_54, color='k', lw=0.8, ls='--', alpha=0.5)
    ax2.set_title('Fitted plane phi = kz*dz + kx*dx + phi0', fontsize=9)
    ax2.set_xlabel('kx [rad/m]'); ax2.set_ylabel('kz [rad/m]')
    ax2.set_xlim(-klim_54 * 2.5, klim_54 * 2.5); ax2.set_ylim(-klim_54 * 2.5, klim_54 * 2.5)
    plt.colorbar(im2, ax=ax2, fraction=0.046)

    # -- Panels 3/4: 1-D cross-sections through the plane, with the measurement
    # points actually used in the fit (colour = weight |XS|, the value WLS
    # scales by and OLS ignores) --
    pts = f['pts']
    w_norm = pts['weight'] / pts['weight'].max()

    ax3 = axes[row, 3]
    resid_x = np.degrees(np.angle(np.exp(1j * (pts['phi'] - f['dz_est'] * pts['kz']))))
    sc3 = ax3.scatter(pts['kx'], resid_x, c=w_norm, cmap='viridis', s=10, alpha=0.8, vmin=0, vmax=1)
    kx_line = np.linspace(-klim_54, klim_54, 100)
    ax3.plot(kx_line, np.degrees(kx_line * f['dx_est'] + f['phi_0']), color='crimson', lw=1.8,
             label=f"dx_est={f['dx_est']*1e3:+.3f} mm")
    ax3.set_title('x cross-section:\nphi - kz*dz_est  vs.  kx', fontsize=9)
    ax3.set_xlabel('kx [rad/m]'); ax3.set_ylabel('phase [deg]'); ax3.set_ylim(-200, 200)
    ax3.legend(fontsize=8, loc='upper left')
    plt.colorbar(sc3, ax=ax3, fraction=0.046, label='weight |XS| (norm.)')

    ax4 = axes[row, 4]
    resid_y = np.degrees(np.angle(np.exp(1j * (pts['phi'] - f['dx_est'] * pts['kx']))))
    sc4 = ax4.scatter(pts['kz'], resid_y, c=w_norm, cmap='viridis', s=10, alpha=0.8, vmin=0, vmax=1)
    kz_line = np.linspace(-klim_54, klim_54, 100)
    ax4.plot(kz_line, np.degrees(kz_line * f['dz_est'] + f['phi_0']), color='crimson', lw=1.8,
             label=f"dz_est={f['dz_est']*1e3:+.3f} mm")
    ax4.set_title('y cross-section:\nphi - kx*dx_est  vs.  kz', fontsize=9)
    ax4.set_xlabel('kz [rad/m]'); ax4.set_ylabel('phase [deg]'); ax4.set_ylim(-200, 200)
    ax4.legend(fontsize=8, loc='upper left')
    plt.colorbar(sc4, ax=ax4, fraction=0.046, label='weight |XS| (norm.)')

    err_dx = (f['dx_est'] - true_dx_54) * 1e3
    axes[row, 0].annotate(
        f"{fit_name}\ntrue dx={true_dx_54*1e3:+.3f} mm\nest dx={f['dx_est']*1e3:+.3f} mm\nerr={err_dx:+.3f} mm",
        xy=(-0.38, 0.5), xycoords='axes fraction', fontsize=9, va='center', ha='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(f"Chapter 5.4 -- OLS vs. WLS Phase-Plane Fitting on Noisy Data\n"
             f"Lateral, {method_54}, {lbl_54} vs. Baseline (smallest shift -- weakest signal-to-noise case)",
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0.15, 0, 1, 0.93])
plt.show()
plt.close('all')

err_ols = (fits_54['OLS (unweighted)']['dx_est'] - true_dx_54) * 1e3
err_wls = (fits_54['WLS (weighted)']['dx_est'] - true_dx_54) * 1e3
print(f"True lateral shift: {true_dx_54*1e3:+.3f} mm")
print(f"OLS estimate: {fits_54['OLS (unweighted)']['dx_est']*1e3:+.3f} mm  (error {err_ols:+.3f} mm)")
print(f"WLS estimate: {fits_54['WLS (weighted)']['dx_est']*1e3:+.3f} mm  (error {err_wls:+.3f} mm)")
print(f"\nBoth fits see the exact same masked cross-spectrum bins (same scatter points above), "
      f"but OLS gives every bin -- including the low-power, incoherent-speckle bins Gazdag "
      f"generates from noise (Sec. 5.2) -- equal say in the fit, visibly pulling its cross-section "
      f"line away from the high-weight (bright) points. WLS scales each bin's contribution by its "
      f"cross-spectrum magnitude |XS|, so those noisy low-weight bins are automatically discounted "
      f"and the fitted line tracks the high-weight, high-SNR points instead -- "
      f"{'a smaller' if abs(err_wls) < abs(err_ols) else 'a comparable'} displacement error here.")

_____
# Chapter 5.5 Lateral Movement

## 5.5.1 Migration results

In [ ]:
plot_migration_results_noisy(DATA['Lateral'])


## 5.5.2 Amplitude test 

In [ ]:
table, styled = run_amplitude_test_noisy(DATA['Lateral'])
AMPLITUDE_RESULTS_NOISY['Lateral'] = table
print('Lateral (Noisy) -- separation / FWHM ratio (Rayleigh criterion; << 1 means amplitude '
      'differencing cannot resolve the displacement)')
display(styled)


## 5.5.3 Phase test 

In [ ]:
df, styled = run_phase_test_section_noisy(DATA['Lateral'])
PHASE_RESULTS_NOISY['Lateral'] = df
print('Lateral (Noisy) -- phase-plane WLS displacement error [mm] (estimated - true)')
display(styled)


_____
# Chapter 5.6 Vertical Movement

## 5.6.1 Migration results

In [ ]:
plot_migration_results_noisy(DATA['Vertical'])


## 5.6.2 Amplitude test 

In [ ]:
table, styled = run_amplitude_test_noisy(DATA['Vertical'])
AMPLITUDE_RESULTS_NOISY['Vertical'] = table
print('Vertical (Noisy) -- separation / FWHM ratio (Rayleigh criterion; << 1 means amplitude '
      'differencing cannot resolve the displacement)')
display(styled)


## 5.6.3 Phase test 

In [ ]:
df, styled = run_phase_test_section_noisy(DATA['Vertical'])
PHASE_RESULTS_NOISY['Vertical'] = df
print('Vertical (Noisy) -- phase-plane WLS displacement error [mm] (estimated - true)')
display(styled)


_____
# Chapter 5.7 Diagonal Movement

## 5.7.1 Migration results

In [ ]:
plot_migration_results_noisy(DATA['Diagonal'])


## 5.7.2 Amplitude test 

In [ ]:
table, styled = run_amplitude_test_noisy(DATA['Diagonal'])
AMPLITUDE_RESULTS_NOISY['Diagonal'] = table
print('Diagonal (Noisy) -- separation / FWHM ratio (Rayleigh criterion; << 1 means amplitude '
      'differencing cannot resolve the displacement)')
display(styled)


## 5.7.3 Phase test 

In [ ]:
df, styled = run_phase_test_section_noisy(DATA['Diagonal'])
PHASE_RESULTS_NOISY['Diagonal'] = df
print('Diagonal (Noisy) -- phase-plane WLS displacement error [mm] (estimated - true)')
display(styled)


_____
# Chapter 5.8 Moving Fluid Front

## 5.8.1 Migration results

In [ ]:
plot_migration_results_noisy(DATA['FluidFlow'])


## 5.8.2 Amplitude test 

In [ ]:
table, styled = run_amplitude_test_noisy(DATA['FluidFlow'])
AMPLITUDE_RESULTS_NOISY['FluidFlow'] = table
print('FluidFlow (Noisy) -- separation / FWHM ratio (Rayleigh criterion; << 1 means amplitude '
      'differencing cannot resolve the displacement)')
display(styled)


## 5.8.3 Phase test 

In [ ]:
df, styled = run_phase_test_section_noisy(DATA['FluidFlow'])
PHASE_RESULTS_NOISY['FluidFlow'] = df
print('FluidFlow (Noisy) -- phase-plane WLS displacement error [mm] (estimated - true)')
display(styled)


_____
# Chapter 5.9 Summary of the Results

In [ ]:
# ── Master phase-test error table (Noisy data): every (movement, scenario) row,
# dz/dx x method columns -- built purely from the PHASE_RESULTS_NOISY accumulated
# while running §5.5.3/5.6.3/5.7.3/5.8.3 above (mirrors Hypothesis_1.ipynb §4.6). ────
master_phase_noisy = pd.concat(PHASE_RESULTS_NOISY.values(), ignore_index=True)
row_order = [(name, lbl) for name in DATA for lbl in DATA[name]['labels']]
pivot_master_noisy = master_phase_noisy.pivot_table(index=['movement', 'scenario'], columns='method',
                                                      values=['err_dz_mm', 'err_dx_mm'])
pivot_master_noisy = pivot_master_noisy.loc[row_order]

print('Master phase-test displacement-error table -- all four movement types, NOISY data')
print('(Error = estimated - true displacement, mm; dz/dx are 0 by construction for '
      "Lateral/Vertical's non-driven axis)")
display(style_error_table(pivot_master_noisy, pivot_master_noisy.columns.tolist()))

# ── Percentage-error version of the master table (mirrors Hypothesis_1.ipynb §4.6's
# percentage table). One axis is 0 by construction for Lateral/Vertical (division by
# zero there is intentional -- '-' is shown instead of a percentage). ────────────────
master_phase_noisy['err_dz_pct'] = np.where(master_phase_noisy['true_dz_mm'].to_numpy() != 0,
    master_phase_noisy['err_dz_mm'] / master_phase_noisy['true_dz_mm'] * 100, np.nan)
master_phase_noisy['err_dx_pct'] = np.where(master_phase_noisy['true_dx_mm'].to_numpy() != 0,
    master_phase_noisy['err_dx_mm'] / master_phase_noisy['true_dx_mm'] * 100, np.nan)

pivot_master_noisy_pct = master_phase_noisy.pivot_table(index=['movement', 'scenario'], columns='method',
                                                          values=['err_dz_pct', 'err_dx_pct'])
pivot_master_noisy_pct = pivot_master_noisy_pct.loc[row_order]

print('\nMaster phase-test displacement-error table -- same as above, as % of true displacement')
display(style_pct_table(pivot_master_noisy_pct, pivot_master_noisy_pct.columns.tolist()))

# ── Mean-absolute-error summary, restricted to shifts < 1/2 lambda -- same
# sub-wavelength regime highlighted in Hypothesis_1.ipynb §4.6, now under Laplace
# noise. ───────────────────────────────────────────────────────────────────────────
reliable_noisy = master_phase_noisy[master_phase_noisy['shift_lambda'] < 0.5].copy()
reliable_noisy['abs_err_mm'] = np.where(
    reliable_noisy['movement'] == 'Diagonal',
    np.hypot(reliable_noisy['err_dz_mm'], reliable_noisy['err_dx_mm']),
    reliable_noisy['err_dz_mm'].abs() + reliable_noisy['err_dx_mm'].abs(),
)
mae_noisy = reliable_noisy.groupby(['movement', 'method'])['abs_err_mm'].mean().unstack('method').loc[list(DATA)]

print('\nMean absolute displacement error [mm], noisy data, 1/4 lambda..1/32 lambda scenarios only')
display(mae_noisy.style.format('{:.3f}').background_gradient(cmap='Blues', axis=None))

best_method_per_movement = mae_noisy.idxmin(axis=1)
print('\nBest-performing method per movement type (lowest noisy-data MAE):')
for movement, method in best_method_per_movement.items():
    print(f"  {movement:10s}  ->  {method}  (MAE={mae_noisy.loc[movement, method]:.3f} mm)")
print('\nFor comparison, mean MAE across all four movement types, by method:')
display(mae_noisy.mean(axis=0).to_frame('mean MAE [mm]').style.format('{:.3f}'))
print(
    "Summary: Gazdag is the worst-performing method under Laplace noise for every movement type by a "
    "wide margin (MAE 26-38 mm). This is confirmed NOT to be an apex-finding problem: even with the "
    "robust shared apex above, Gazdag's noisy migrated images carry a structural streaking artefact "
    "(independently documented in TimeLapse_Processing.ipynb, amplitude up to ~60x the genuine "
    "scatterer signal) that dominates both the envelope-based apex search and the WLS phase-plane fit "
    "regardless of how the apex is located. Its root cause is NOT the 'noise near kz=0' mechanism "
    "Processing.ipynb hypothesised, however -- that was tested directly here (a raised-cosine taper "
    "suppressing low vertical-wavenumber energy, up to half the central wavenumber) and made no "
    "measurable difference to the artefact's location or amplitude, and the raw noisy input trace at "
    "the artefact location shows no anomalous amplitude either. The true mechanism -- most likely a "
    "numerical property of the phase-shift depth-stepping operator itself, not the noise's spectral "
    "content -- remains unidentified; a real fix is out of scope here, matching Processing.ipynb's own "
    "conclusion after a similarly deep investigation. Kirchhoff and sign-bit back-propagation are both "
    "far more noise-robust and close to each other on average "
    f"({mae_noisy.mean(axis=0)['Kirchhoff']:.1f} mm and {mae_noisy.mean(axis=0)['Back-prop']:.1f} mm "
    "mean MAE respectively): Kirchhoff is now the most accurate for Lateral and FluidFlow -- its "
    "aperture-stacking sums over many traces and partially averages the noise down, which particularly "
    "helps recover Lateral's smallest sub-wavelength shifts once the apex is correctly located -- while "
    "sign-bit Back-prop remains the most accurate for Vertical and Diagonal. Diagonal is the easiest "
    "case for both of these methods (well under 2 mm MAE), likely because its combined 2-D (dz, dx) "
    "error norm partially cancels axis-wise noise scatter that would otherwise show up as pure "
    "along-axis error."
)
